## Collaborative Filtering

This notebook trains an ALS (Alternating Least Squares) model with weighted confidence for implicit feedback.  
The goal is to learn item embeddings that capture collaborative patterns ("players who played X also played Y") from steam user-game interactions.

In [11]:
import numpy as np
import pandas as pd
import json
from pathlib import Path
from scipy import sparse
from implicit.als import AlternatingLeastSquares

PROJECT_ROOT = Path.cwd().parent

df = pd.read_csv(
    PROJECT_ROOT / "data" / "raw" / "recommendations.csv",
    usecols=["user_id", "app_id", "is_recommended", "hours"]
)

print(f"Total interactions: {len(df):,}")
print(f"Unique users: {df['user_id'].nunique():,}")
print(f"Unique games: {df['app_id'].nunique():,}")

Total interactions: 41,154,794
Unique users: 13,781,059
Unique games: 37,610


### Data Cleaning

Remove zero-hour interactions (no real engagement) and duplicate user-game pairs.

In [12]:
df = df[df["hours"] > 0]
df = df.drop_duplicates(subset=["user_id", "app_id"])

print(f"After cleaning: {len(df):,} interactions")

After cleaning: 40,978,444 interactions


### Iterative K-Core Filtering

Single-pass filtering (remove users < 5, games < 100) leaves orphans: after removing games, some users drop below the threshold again.  
Iterative k-core repeats until both constraints hold simultaneously, producing a denser and cleaner matrix.

In [13]:
MIN_USER_INTERACTIONS = 5
MIN_GAME_INTERACTIONS = 100

prev_len = 0
iteration = 0

while len(df) != prev_len:
    prev_len = len(df)
    iteration += 1

    user_counts = df["user_id"].value_counts()
    df = df[df["user_id"].isin(user_counts[user_counts >= MIN_USER_INTERACTIONS].index)]

    game_counts = df["app_id"].value_counts()
    df = df[df["app_id"].isin(game_counts[game_counts >= MIN_GAME_INTERACTIONS].index)]

    print(f"  Iteration {iteration}: {len(df):,} interactions, {df['user_id'].nunique():,} users, {df['app_id'].nunique():,} games")

num_users = df["user_id"].nunique()
num_games = df["app_id"].nunique()
sparsity = 1 - len(df) / (num_users * num_games)

print(f"\nFinal: {len(df):,} interactions | {num_users:,} users | {num_games:,} games | Sparsity: {sparsity:.4f}")

  Iteration 1: 21,626,828 interactions, 1,903,219 users, 10,886 games
  Iteration 2: 21,506,086 interactions, 1,873,400 users, 10,814 games
  Iteration 3: 21,504,237 interactions, 1,872,962 users, 10,813 games
  Iteration 4: 21,504,225 interactions, 1,872,959 users, 10,813 games
  Iteration 5: 21,504,225 interactions, 1,872,959 users, 10,813 games

Final: 21,504,225 interactions | 1,872,959 users | 10,813 games | Sparsity: 0.9989


### Feature Engineering

- `rating`: binary preference (1 = recommended, 0 = not)  
- `hours_log`: log-transformed playtime, outliers capped at 99th percentile  
- `confidence`: Hu et al. (2008) formulation — `1 + α × hours_log`. Higher playtime = higher confidence that the user truly likes the game

In [14]:
ALPHA = 40

df["rating"] = df["is_recommended"].astype(int)
df["hours_log"] = np.log1p(df["hours"].clip(upper=df["hours"].quantile(0.99)))
df["confidence"] = 1 + ALPHA * df["hours_log"]

df_cf = df[["user_id", "app_id", "rating", "hours_log", "confidence"]].copy()
print(df_cf.head(10))
print(f"\nConfidence range: {df_cf['confidence'].min():.1f} - {df_cf['confidence'].max():.1f}")

    user_id   app_id  rating  hours_log  confidence
0     51580   975370       1   3.618993  145.759733
12   161081  1938090       1   3.864931  155.597256
16    76583   635260       1   5.181784  208.271342
21   664486  1938090       1   2.128232   86.129268
22    22793   534380       1   3.728100  150.124007
23   271318   518790       1   2.397895   96.915811
27   433335    42700       0   1.931521   78.260856
32   912612   438100       1   2.208274   89.330977
36   337740  1794680       1   4.039536  162.581453
41   763450   359550       1   5.121580  205.863208

Confidence range: 4.8 - 268.3


### Sparse Matrix Construction

ALS expects a sparse `user × item` matrix. The values are confidence-weighted preferences:  
`matrix[u, i] = confidence × rating` — so negative reviews get 0 and positive reviews get their confidence value.

In [15]:
# Map original IDs to contiguous indices
user_ids = df_cf["user_id"].unique()
item_ids = df_cf["app_id"].unique()

user_id_to_idx = {uid: idx for idx, uid in enumerate(user_ids)}
item_id_to_idx = {iid: idx for idx, iid in enumerate(item_ids)}
idx_to_item_id = {idx: int(iid) for iid, idx in item_id_to_idx.items()}
idx_to_user_id = {idx: int(uid) for uid, idx in user_id_to_idx.items()}

rows = df_cf["user_id"].map(user_id_to_idx).values
cols = df_cf["app_id"].map(item_id_to_idx).values
values = (df_cf["confidence"] * df_cf["rating"]).values

user_item = sparse.csr_matrix(
    (values, (rows, cols)),
    shape=(len(user_ids), len(item_ids)),
)

print(f"User-Item matrix: {user_item.shape}")
print(f"Non-zero entries: {user_item.nnz:,}")
print(f"Density: {user_item.nnz / (user_item.shape[0] * user_item.shape[1]):.6f}")

User-Item matrix: (1872959, 10813)
Non-zero entries: 21,504,225
Density: 0.001062


### Train / Test Split

Random 80/20 split. We mask 20% of observed interactions as test data, keeping the same matrix shape. The model trains on the remaining 80%.

In [16]:
np.random.seed(42)

# Get indices of all non-zero entries
nonzero_rows, nonzero_cols = user_item.nonzero()
n_interactions = len(nonzero_rows)

# Randomly select 20% for test
test_mask = np.random.random(n_interactions) < 0.2

# Build test matrix
test_data = np.array(user_item[nonzero_rows[test_mask], nonzero_cols[test_mask]]).flatten()
test_matrix = sparse.csr_matrix(
    (test_data, (nonzero_rows[test_mask], nonzero_cols[test_mask])),
    shape=user_item.shape,
)

# Build train matrix (remove test entries from original)
train_data = np.array(user_item[nonzero_rows[~test_mask], nonzero_cols[~test_mask]]).flatten()
train_matrix = sparse.csr_matrix(
    (train_data, (nonzero_rows[~test_mask], nonzero_cols[~test_mask])),
    shape=user_item.shape,
)

print(f"Train interactions: {train_matrix.nnz:,}")
print(f"Test interactions:  {test_matrix.nnz:,}")

Train interactions: 14,648,795
Test interactions:  3,658,730


### ALS Training

Implict's ALS alternates between fixing user factors and solving for item factors, then vice versa. The confidence values tell the model how much to trust each observation.

In [17]:
model = AlternatingLeastSquares(
    factors=128,
    iterations=15,
    regularization=0.01,
    random_state=42,
)

model.fit(train_matrix)

/home/axel/grec/.venv/lib/python3.12/site-packages/implicit/cpu/als.py:95: RuntimeWarning: OpenBLAS is configured to use 16 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
100%|██████████| 15/15 [01:29<00:00,  5.98s/it]


### Evaluation

- **Precision@10**: of the top 10 recommended items, what fraction appear in the held-out test set?  
- **MAP@10**: same idea but rewards models that rank test items higher in the list

In [20]:
K = 10
N_EVAL_USERS = 50_000  # sample for speed

rng = np.random.default_rng(42)
test_users = np.array(list(set(test_matrix.nonzero()[0])))
eval_users = rng.choice(test_users, size=min(N_EVAL_USERS, len(test_users)), replace=False)

precisions = []
avg_precisions = []

for user_idx in eval_users:
    test_items = set(test_matrix[user_idx].nonzero()[1])
    if not test_items:
        continue

    rec_indices, _ = model.recommend(
        user_idx, train_matrix[user_idx], N=K, filter_already_liked_items=True
    )

    # Precision@K
    hits = [1 if r in test_items else 0 for r in rec_indices]
    precisions.append(sum(hits) / K)

    # Average Precision@K
    cum_hits = 0
    ap = 0.0
    for rank, h in enumerate(hits, 1):
        cum_hits += h
        if h:
            ap += cum_hits / rank
    avg_precisions.append(ap / min(len(test_items), K))

print(f"Evaluated on {len(precisions):,} users")
print(f"Precision@{K}: {np.mean(precisions):.4f}")
print(f"MAP@{K}:       {np.mean(avg_precisions):.4f}")

Evaluated on 50,000 users
Precision@10: 0.0223
MAP@10:       0.0398


### Sample Recommendations

Quick sanity check: pick a random user and see what the model recommends vs what they actually played.

In [21]:
sample_user_idx = np.random.randint(0, len(user_ids))
sample_user_id = idx_to_user_id[sample_user_idx]

# What they already played
played_indices = user_item[sample_user_idx].nonzero()[1]
played_app_ids = [idx_to_item_id[i] for i in played_indices]
print(f"User {sample_user_id} played {len(played_app_ids)} games: {played_app_ids[:10]}...")

# Model recommendations
rec_indices, rec_scores = model.recommend(
    sample_user_idx,
    user_item[sample_user_idx],
    N=10,
    filter_already_liked_items=True,
)

print("\nTop 10 recommendations:")
for idx, score in zip(rec_indices, rec_scores):
    print(f"  app_id={idx_to_item_id[idx]:>8}  score={score:.4f}")

User 13632835 played 5 games: [304390, 3590, 359320, 779340, 872410]...

Top 10 recommendations:
  app_id= 1189630  score=0.5637
  app_id= 1468810  score=0.5425
  app_id=  875210  score=0.5344
  app_id=  952860  score=0.5122
  app_id=  594570  score=0.4939
  app_id=     440  score=0.4526
  app_id=  882790  score=0.4502
  app_id= 1559390  score=0.4453
  app_id= 1543030  score=0.4384
  app_id=  628070  score=0.4351


### Save Artifacts

Export item factors and ID mappings. These are what the backend loads to serve recommendations. The user factors are not needed since app users build pseudo-user vectors from their library.

In [ ]:
output_dir = PROJECT_ROOT / "data" / "processed" / "collaborative"
output_dir.mkdir(parents=True, exist_ok=True)

# Item factors: (n_items, 128) — the core artifact for recommendations
np.save(output_dir / "item_factors.npy", model.item_factors)

# ID mappings: app_id <-> matrix index
with open(output_dir / "item_id_mapping.json", "w") as f:
    json.dump({
        "app_id_to_idx": {str(k): v for k, v in item_id_to_idx.items()},
        "idx_to_app_id": {str(k): v for k, v in idx_to_item_id.items()},
    }, f)

print(f"Saved item_factors.npy: {model.item_factors.shape}")
print(f"Saved item_id_mapping.json: {len(item_id_to_idx)} items")
print(f"Output: {output_dir}")